<a href="https://colab.research.google.com/github/CFRDeusa/Teste/blob/master/C%C3%B3pia_de_Untitled3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tkinter as tk

# --- Variáveis do Processo (Memória) ---
motor_ligado = False
nivel_tanque = 0  # Começa vazio (0 a 100)
direcao = 0       # 1 = Enchendo, -1 = Esvaziando, 0 = Parado
# --- Lógica do CLP (Ciclo Automático) ---
def ciclo_scan():
    global nivel_tanque, motor_ligado

    # Simula a física (Se motor ligado, nível sobe)
    if motor_ligado:
        if nivel_tanque < 100:
            nivel_tanque += 1  # Velocidade de enchimento
        else:
            motor_ligado = False # Desliga sozinho se encher (Sensor de Nível Alto)
            lbl_aviso.config(text="ALERTA: Tanque Cheio!", fg="red")

    # Se estiver esvaziando (dreno aberto)
    if direcao == -1:
         if nivel_tanque > 0:
            nivel_tanque -= 1

    # --- Atualiza o Desenho (Animação) ---
    # A mágica acontece aqui: mudamos a altura do retângulo azul
    # Coordenadas: x1, y1 (topo variavel), x2, y2 (base fixa)
    altura_pixel = 250 - (nivel_tanque * 2) # Converte % em pixels
    canvas.coords(concreto, 50, altura_pixel, 150, 250)

    # Atualiza texto de porcentagem
    lbl_nivel.config(text=f"Nível: {nivel_tanque}%")

    # Configura o Python para rodar essa função de novo em 50ms
    root.after(50, ciclo_scan)

# --- Comandos dos Botões ---
def ligar_bomba():
    global motor_ligado, direcao
    motor_ligado = True
    direcao = 1
    lbl_aviso.config(text="Status: Bombeando Concreto...", fg="green")

def abrir_dreno():
    global motor_ligado, direcao
    motor_ligado = False
    direcao = -1
    lbl_aviso.config(text="Status: Drenando Silo...", fg="orange")

def parar_tudo():
    global motor_ligado, direcao
    motor_ligado = False
    direcao = 0
    lbl_aviso.config(text="Status: Parado", fg="black")

# --- Interface Gráfica ---
root = tk.Tk()
root.title("Simulador de Silo de Concreto")
root.geometry("400x500")

lbl_titulo = tk.Label(root, text="Controle de Bomba", font=("Arial", 16, "bold"))
lbl_titulo.pack(pady=10)

canvas = tk.Canvas(root, width=200, height=300, bg="white")
canvas.pack()

# Desenha o Tanque (Borda fixa)
canvas.create_rectangle(50, 50, 150, 250, outline="black", width=3)
# Desenha o Concreto (Começa vazio, lá embaixo)
concreto = canvas.create_rectangle(50, 250, 150, 250, fill="gray", outline="")

lbl_nivel = tk.Label(root, text="Nível: 0%", font=("Arial", 12, "bold"))
lbl_nivel.pack()

lbl_aviso = tk.Label(root, text="Status: Aguardando...", fg="blue")
lbl_aviso.pack(pady=5)

frame_btn = tk.Frame(root)
frame_btn.pack(pady=20)

btn_ligar = tk.Button(frame_btn, text="LIGAR BOMBA", bg="#90ee90", command=ligar_bomba)
btn_ligar.grid(row=0, column=0, padx=5)

btn_parar = tk.Button(frame_btn, text="PARAR", bg="#f0f0f0", command=parar_tudo)
btn_parar.grid(row=0, column=1, padx=5)

btn_dreno = tk.Button(frame_btn, text="ABRIR DRENO", bg="#add8e6", command=abrir_dreno)
btn_dreno.grid(row=0, column=2, padx=5)

# Inicia o Ciclo (O coração do simulador)
ciclo_scan()

root.mainloop()

TclError: no display name and no $DISPLAY environment variable

### Simulação de Silo de Concreto com `ipywidgets`

Este código adapta a interface gráfica original baseada em Tkinter para usar `ipywidgets`, permitindo que a simulação seja executada interativamente no ambiente do Google Colab. As principais mudanças incluem:

*   **Substituição de Tkinter por `ipywidgets`**: `tk.Tk`, `tk.Label`, `tk.Canvas`, `tk.Button` foram substituídos por `ipywidgets.VBox`, `ipywidgets.Label`, `ipywidgets.IntProgress`, `ipywidgets.Button` e `ipywidgets.HTML`.
*   **Representação do Silo**: O `tk.Canvas` com o retângulo do concreto foi substituído por um `ipywidgets.IntProgress` com orientação vertical, que visualmente representa o nível de enchimento do silo.
*   **Ciclo de Scan**: A chamada `root.after(50, ciclo_scan)` (que agendava a próxima atualização da UI no Tkinter) foi substituída por loops `while` dentro das funções dos botões (`on_ligar_bomba_clicked`, `on_abrir_dreno_clicked`). Esses loops usam `time.sleep(0.05)` para simular o atraso e atualizam os widgets diretamente.

**Observação Importante**: Durante os processos de "LIGAR BOMBA" ou "ABRIR DRENO", o loop de simulação bloqueia temporariamente a execução do kernel. Isso significa que a interface pode parecer "congelada" durante o enchimento/esvaziamento, e você não poderá clicar em outros botões até que a ação atual termine ou seja interrompida pelo botão "PARAR" (que define `motor_ligado = False` e `direcao = 0`, encerrando o loop ativo).

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

# --- Variáveis do Processo (Memória) ---
motor_ligado = False
nivel_tanque = 0  # Começa vazio (0 a 100)
direcao = 0       # 1 = Enchendo, -1 = Esvaziando, 0 = Parado

# --- Widgets de Interface ---
lbl_titulo = widgets.Label("Simulador de Silo de Concreto")
lbl_titulo.layout.font_size = '16px'
lbl_titulo.layout.font_weight = 'bold'
lbl_titulo.layout.align_self = 'center'

# Usamos IntProgress para simular o tanque
# orientation='vertical' para simular um silo
# description='Nível do Tanque:' para o rótulo
prog_bar_tanque = widgets.IntProgress(
    value=nivel_tanque,
    min=0,
    max=100,
    description='Nível:',
    orientation='vertical',
    bar_style='info', # Cor azul
    layout=widgets.Layout(height='200px', width='50px', border='solid 1px black') # Estilo do silo
)

lbl_nivel = widgets.Label(f"Nível: {nivel_tanque}%")
lbl_aviso = widgets.HTML(value="<span style='color: blue;'>Status: Aguardando...</span>")

btn_ligar = widgets.Button(description="LIGAR BOMBA", button_style='success')
btn_parar = widgets.Button(description="PARAR", button_style='info')
btn_dreno = widgets.Button(description="ABRIR DRENO", button_style='warning')

# --- Funções de Atualização e Lógica ---
def _update_display_widgets():
    """Atualiza o estado visual dos widgets com base nas variáveis globais."""
    prog_bar_tanque.value = nivel_tanque
    lbl_nivel.value = f"Nível: {nivel_tanque}%"

def ciclo_scan_step():
    """Executa um único passo da simulação e atualiza os widgets."""
    global nivel_tanque, motor_ligado, direcao

    # Simula a física (Se motor ligado e enchendo, nível sobe)
    if motor_ligado and direcao == 1:
        if nivel_tanque < 100:
            nivel_tanque += 1
        else:
            motor_ligado = False # Desliga sozinho se encher (Sensor de Nível Alto)
            lbl_aviso.value = "<span style='color: red;'>ALERTA: Tanque Cheio!</span>"
            direcao = 0 # Para a direção
    elif direcao == -1: # Se estiver esvaziando (dreno aberto)
        if nivel_tanque > 0:
            nivel_tanque -= 1
        else:
            direcao = 0 # Para de esvaziar quando vazio
            lbl_aviso.value = "<span style='color: blue;'>Status: Silo Vazio.</span>"

    _update_display_widgets()

# --- Comandos dos Botões (Callbacks) ---
def on_ligar_bomba_clicked(b):
    global motor_ligado, direcao, nivel_tanque
    motor_ligado = True
    direcao = 1
    lbl_aviso.value = "<span style='color: green;'>Status: Bombeando Concreto...</span>"
    # Simula o enchimento continuamente
    while motor_ligado and nivel_tanque < 100 and direcao == 1:
        ciclo_scan_step()
        time.sleep(0.05) # Pequeno atraso para simular o tempo
    # Garante que o status final seja exibido se o loop terminar
    if nivel_tanque == 100 and direcao == 1: # Se terminou de encher naturalmente
        lbl_aviso.value = "<span style='color: red;'>ALERTA: Tanque Cheio!</span>"
    _update_display_widgets() # Atualiza o display uma última vez para sincronizar

def on_abrir_dreno_clicked(b):
    global motor_ligado, direcao, nivel_tanque
    motor_ligado = False # Dreno aberto, bomba desligada
    direcao = -1
    lbl_aviso.value = "<span style='color: orange;'>Status: Drenando Silo...</span>"
    # Simula o esvaziamento continuamente
    while nivel_tanque > 0 and direcao == -1:
        ciclo_scan_step()
        time.sleep(0.05) # Pequeno atraso para simular o tempo
    # Garante que o status final seja exibido se o loop terminar
    if nivel_tanque == 0 and direcao == -1: # Se terminou de esvaziar naturalmente
        lbl_aviso.value = "<span style='color: blue;'>Status: Silo Vazio.</span>"
    _update_display_widgets() # Atualiza o display uma última vez para sincronizar

def on_parar_tudo_clicked(b):
    global motor_ligado, direcao
    motor_ligado = False
    direcao = 0
    lbl_aviso.value = "<span style='color: black;'>Status: Parado</span>"
    _update_display_widgets() # Atualiza para garantir que a UI reflete o estado parado

# Atribui os callbacks aos botões
btn_ligar.on_click(on_ligar_bomba_clicked)
btn_parar.on_click(on_parar_tudo_clicked)
btn_dreno.on_click(on_abrir_dreno_clicked)

# --- Layout da Interface ---
# Layout para o silo e o nível
silo_display = widgets.VBox([
    prog_bar_tanque,
    lbl_nivel
], layout=widgets.Layout(align_items='center', margin='10px 0'))

# Layout para os botões
button_box = widgets.HBox([btn_ligar, btn_parar, btn_dreno], layout=widgets.Layout(justify_content='space-around', width='auto'))

# Layout geral
ui = widgets.VBox([
    lbl_titulo,
    silo_display,
    lbl_aviso,
    button_box
], layout=widgets.Layout(
    border='solid 1px lightgray',
    padding='10px',
    width='400px', # Ajustado para melhor visualização
    align_items='center'
))

# Exibe a interface
display(ui)
